# Budgeted metamorphic search study

This notebook runs only the retained budgeted experiments. The plots report actual wall-clock execution time in seconds. Timeout groups are kept in the CSV files, but boxes are omitted when more than half of the corresponding runs time out.

In [ ]:
from pathlib import Path
import sys

candidate = Path('subprojects/metamorphic-2')
ROOT = candidate.resolve() if candidate.exists() else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

BUDGET = 2.0
SIZES = (10, 20, 50, 100)
TRIALS = 3
SEED = 7
NSGA_POPULATION = 60
NSGA_GENERATIONS = 30

budget_dir = ROOT / 'budget_results'
results_dir = ROOT / 'results'
figures_dir = ROOT / 'figures'
budget_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

ROOT

## Fixed-budget N study

This runs all four methods on the lift and AV mock case studies at `N = 10, 20, 50, 100`, with each method-instance run terminated at `BUDGET` seconds. The generated figure has three shared-x panels: execution time, found-set cardinality, and robustness.

In [ ]:
from IPython.display import Image, display
from budget_study import (
    perform_budget_anova,
    plot_budget_results,
    run_budget_study,
    summarize as summarize_budget,
)

budget_results = run_budget_study(
    budget=BUDGET,
    sizes=SIZES,
    trials=TRIALS,
    seed=SEED,
    nsga_population=NSGA_POPULATION,
    nsga_generations=NSGA_GENERATIONS,
)
budget_summary = summarize_budget(budget_results)

budget_results.to_csv(budget_dir / 'budget_runs.csv', index=False)
budget_summary.to_csv(budget_dir / 'budget_summary.csv', index=False)
perform_budget_anova(budget_results, completed_only=True, budget=BUDGET).to_csv(
    budget_dir / 'budget_anova_completed_only.csv', index=False
)
perform_budget_anova(budget_results, completed_only=False, budget=BUDGET).to_csv(
    budget_dir / 'budget_anova_timeout_as_budget.csv', index=False
)
plot_budget_results(budget_results, BUDGET, budget_dir / 'budget_times.png', sizes=SIZES)

display(budget_summary)
display(Image(filename=str(budget_dir / 'budget_times.png')))

## Budgeted bundle-size study

This runs the lift-only bundle-size study for `B = 1, 2, 5` under the same wall-clock budget. It produces one runtime plot per method.

In [ ]:
from rq2_budget_bundle_study import (
    create_method_plots,
    run_budgeted_rq2,
    summarize as summarize_budgeted_rq2,
)

rq2_results = run_budgeted_rq2(
    sizes=SIZES,
    bundle_sizes=(1, 2, 5),
    budget=BUDGET,
    trials=TRIALS,
    seed=SEED,
    nsga_population=NSGA_POPULATION,
    nsga_generations=NSGA_GENERATIONS,
)
rq2_summary = summarize_budgeted_rq2(rq2_results)

rq2_results.to_csv(results_dir / 'rq2_budget_bundle_results.csv', index=False)
rq2_summary.to_csv(results_dir / 'rq2_budget_bundle_summary.csv', index=False)
create_method_plots(rq2_results, figures_dir, BUDGET, sizes=SIZES, bundle_sizes=(1, 2, 5))

display(rq2_summary)
for name in [
    'lift_rq2_budget_bfs.png',
    'lift_rq2_budget_approx_best_first.png',
    'lift_rq2_budget_nsga2.png',
    'lift_rq2_budget_best_first_shrink.png',
]:
    display(Image(filename=str(figures_dir / name)))